# Task 2 — Label Analysis

**Author:** Dat Nguyen

**Questions answered:**
1. How many POOR vs FAIR vs GOOD hours are there overall?
2. Is the dataset class-imbalanced, and how badly?
3. Does each break get a different rating distribution?

**Approach:** Each raw feature (wave height, period, wind speed, wind direction, tide) is labeled independently on its own scale — POOR / FAIR / GOOD. No single bad feature kills the whole session. The overall surf rating is the Surfline label that the model learns to predict. For now it is derived by averaging the feature scores, since real Surfline labels (`fetch_labels.py`) are not yet implemented.

> Swap in real Surfline labels at Section 2 when available — the rest of the notebook runs unchanged.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

RATINGS      = ['GOOD', 'FAIR', 'POOR']
COLORS       = {'GOOD': '#2ecc71', 'FAIR': '#f39c12', 'POOR': '#e74c3c'}
BREAKS       = ['blacks', 'la_jolla_shores', 'pb_point']
BREAK_LABELS = {'blacks': 'Blacks Beach', 'la_jolla_shores': 'La Jolla Shores', 'pb_point': 'PB Point'}

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('ready')

## 1. Feature labelers

Each function looks at one feature in isolation and returns POOR / FAIR / GOOD for that feature only. A bad tide label does not affect the wave height label.

In [ ]:
def label_wvht(m):
    """
    Wave height (meters).
    POOR  : < 0.5m (flat, nothing to ride) or > 3.0m (dangerous for most)
    FAIR  : 0.5–1.0m (small, marginal) or 2.5–3.0m (large, doable but tricky)
    GOOD  : 1.0–2.5m — ideal SD range, head-high to a bit overhead
    """
    ft = m * 3.281
    if ft < 1.6 or ft > 9.8:   return 'POOR'
    if ft < 3.3 or ft > 8.2:   return 'FAIR'
    return 'GOOD'


def label_dpd(s):
    """
    Dominant period (seconds) — most important quality indicator.
    POOR  : < 8s — local wind chop, waves crashing into each other, no power
    FAIR  : 8–12s — moderate, some organization but not groundswell
    GOOD  : 12–18s — proper groundswell, powerful and organized
            > 18s also GOOD — exceptional, forgiving of other weak factors
    """
    if s < 8:    return 'POOR'
    if s < 12:   return 'FAIR'
    return 'GOOD'


def label_wind_speed(mph, dpd):
    """
    Wind speed (mph).
    POOR  : > 15 mph — too strong, degrades wave face
            Exception: period > 18s can handle up to 18 mph
            (powerful swell holds up even in stronger wind)
    FAIR  : 10–15 mph — manageable but noticeable
    GOOD  : 0–10 mph — light, minimal impact on wave quality
    """
    limit = 18 if dpd > 18 else 15
    if mph > limit:   return 'POOR'
    if mph > 10:      return 'FAIR'
    return 'GOOD'


def label_wind_dir(degrees):
    """
    Wind direction relative to the beach.
    All three SD breaks face roughly west, so East wind is offshore for all of them.
    Source: surf-forecast.com and deepswell.com list East as best wind for Blacks,
    La Jolla Shores, and PB Point.

    GOOD  : 45°–135° (NE through SE, centered on East 90°)
            Wind blows away from the beach — grooms the wave face clean
    FAIR  : within 90° of East but outside the 45° cone — cross-shore, not ideal
    POOR  : outside 135° from East — clearly onshore, pushes waves over early
    """
    delta = abs((degrees - 90 + 180) % 360 - 180)
    if delta <= 45:    return 'GOOD'
    if delta <= 90:    return 'FAIR'
    return 'POOR'


# Preferred tide window per break (feet)
TIDE_WIN = {
    'blacks':          (0.5, 3.5),   # canyon focuses swell at lower tides
    'la_jolla_shores': (2.5, 5.0),   # sandy beach needs enough water depth
    'pb_point':        (1.0, 3.5),   # mixed sand/reef, moderate depth
}

def label_tide(ft, brk):
    """
    Tide height (feet) relative to each break's preferred window.
    GOOD  : inside preferred window
    FAIR  : within 0.5 ft outside the window — close enough
    POOR  : more than 0.5 ft outside — wrong depth for this break
    """
    lo, hi = TIDE_WIN[brk]
    if lo <= ft <= hi:                         return 'GOOD'
    if lo - 0.5 <= ft <= hi + 0.5:            return 'FAIR'
    return 'POOR'


print('Feature labelers defined.')
# Quick sanity checks
assert label_wvht(1.5)  == 'GOOD'
assert label_wvht(0.1)  == 'POOR'
assert label_dpd(16)    == 'GOOD'
assert label_dpd(6)     == 'POOR'
assert label_wind_dir(90)  == 'GOOD'   # due east = offshore
assert label_wind_dir(270) == 'POOR'   # due west = onshore
assert label_tide(2.0, 'blacks') == 'GOOD'
assert label_tide(6.0, 'blacks') == 'POOR'
print('All assertions passed.')

## 2. Build dataset

Generate 180 days of synthetic SD conditions, apply each feature labeler independently, then derive an overall rating by averaging the feature scores. This overall rating is the stand-in for Surfline labels until `fetch_labels.py` is implemented.

In [ ]:
SCORE = {'GOOD': 2, 'FAIR': 1, 'POOR': 0}

def overall_rating(scores):
    """Average feature scores → overall session rating."""
    avg = np.mean([SCORE[s] for s in scores])
    if avg >= 1.5:   return 'GOOD'
    if avg >= 0.75:  return 'FAIR'
    return 'POOR'


# --- Synthetic conditions: 180 days (SD winter/spring Oct 2025 – Mar 2026) ---
rng    = np.random.default_rng(42)
hours  = pd.date_range('2025-10-01', periods=180 * 24, freq='h', tz='UTC')
n      = len(hours)
pac    = (hours.hour - 8) % 24
month  = hours.month
winter = (month >= 10) | (month <= 3)

# Wave height: larger in winter (more NW groundswell)
wvht = np.clip(rng.lognormal(np.where(winter, 0.45, 0.15), 0.55, n), 0.2, 4.5)

# Period: 30% groundswell (12–22s), 70% short-period wind swell (4–12s)
dpd = np.where(
    rng.random(n) < 0.30,
    rng.normal(15, 2, n).clip(12, 22),
    rng.normal(8.5, 1.5, n).clip(4, 12)
)

# Wind speed: calm at dawn, builds to onshore sea breeze by afternoon
base_spd = 5 + 14 * np.clip(np.sin((pac - 5) * np.pi / 12), 0, 1)
wind_mph = np.clip(rng.normal(base_spd, 3.5, n), 0, 40)

# Wind direction: easterly offshore at dawn, westerly onshore by afternoon
wind_dir = np.where(
    (pac >= 4) & (pac < 11),  rng.normal(80,  25, n),
    np.where(
    (pac >= 12) & (pac < 20), rng.normal(270, 30, n),
                              rng.uniform(0, 360, n))
) % 360

# Tide: semi-diurnal (two highs + two lows every ~12.4 hours)
tide_ft = np.clip(
    2.5 + 2.0 * np.sin(hours.hour * 2 * np.pi / 12.4) + rng.normal(0, 0.3, n),
    0, 6.5
)

base = pd.DataFrame(dict(
    hour_utc=hours, pacific_hour=pac, month=month,
    wvht=wvht, dpd=dpd, wind_mph=wind_mph, wind_dir=wind_dir, tide_ft=tide_ft
))

# Build per-break dataset with independent feature labels
offsets = {'blacks': -10, 'la_jolla_shores': 10, 'pb_point': 0}
frames  = []
for brk, offset in offsets.items():
    d = base.copy()
    d['break'] = brk
    d['wind_dir_break'] = (d['wind_dir'] + offset) % 360

    # Label each feature independently
    d['wvht_label']       = d['wvht'].apply(label_wvht)
    d['dpd_label']        = d['dpd'].apply(label_dpd)
    d['wind_speed_label'] = d.apply(lambda r: label_wind_speed(r.wind_mph, r.dpd), axis=1)
    d['wind_dir_label']   = d['wind_dir_break'].apply(label_wind_dir)
    d['tide_label']       = d.apply(lambda r: label_tide(r.tide_ft, brk), axis=1)

    # Overall rating = average of all feature scores
    FEAT_COLS = ['wvht_label', 'dpd_label', 'wind_speed_label', 'wind_dir_label', 'tide_label']
    d['rating'] = d[FEAT_COLS].apply(lambda row: overall_rating(row.tolist()), axis=1)
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
print(f'Dataset: {len(df):,} rows  ({df["break"].nunique()} breaks × {180*24:,} hours)')
print('\nFeature label sample:')
df[['break'] + FEAT_COLS + ['rating']].head()

## 3. Feature label distributions

How often is each individual feature POOR / FAIR / GOOD across all hours?

In [ ]:
FEAT_META = {
    'wvht_label':       'Wave Height',
    'dpd_label':        'Period (DPD)',
    'wind_speed_label': 'Wind Speed',
    'wind_dir_label':   'Wind Direction',
    'tide_label':       'Tide',
}

fig, axes = plt.subplots(1, len(FEAT_META), figsize=(16, 4), sharey=False)

for ax, (col, title) in zip(axes, FEAT_META.items()):
    counts = df[col].value_counts().reindex(RATINGS, fill_value=0)
    pct    = (counts / counts.sum() * 100).round(1)
    bars   = ax.bar(counts.index, counts.values,
                    color=[COLORS[r] for r in counts.index],
                    edgecolor='white', linewidth=0.5, width=0.6)
    for bar, p in zip(bars, pct.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + counts.max() * 0.02,
                f'{p:.0f}%', ha='center', va='bottom', fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('# hours' if ax == axes[0] else '')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='x', labelsize=8)

plt.suptitle('Individual Feature Label Distributions (all breaks, 180 days)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Overall POOR / FAIR / GOOD distribution

In [ ]:
overall     = df['rating'].value_counts().reindex(RATINGS, fill_value=0)
overall_pct = (overall / overall.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

bars = axes[0].bar(
    overall.index, overall.values,
    color=[COLORS[r] for r in overall.index],
    edgecolor='white', linewidth=0.8, width=0.55
)
for bar, cnt, pct in zip(bars, overall.values, overall_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + overall.max() * 0.01,
                 f'{cnt:,}\n({pct}%)', ha='center', va='bottom', fontsize=9)
axes[0].set_ylabel('# hours')
axes[0].set_title('Overall label counts — all 3 breaks combined')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

axes[1].pie(
    overall.values,
    labels=[f'{r}  {p}%' for r, p in zip(overall.index, overall_pct.values)],
    colors=[COLORS[r] for r in overall.index],
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
axes[1].set_title('Label share')

plt.suptitle('POOR / FAIR / GOOD Distribution (180-day SD winter/spring)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print(overall_pct.rename('share %').to_string())

## 5. Class imbalance

In [ ]:
majority = overall.idxmax()
minority = overall.idxmin()
ratio    = overall.max() / overall.min()

print(f'Majority class : {majority}  ({overall[majority]:,} hrs, {overall_pct[majority]}%)')
print(f'Minority class : {minority}  ({overall[minority]:,} hrs, {overall_pct[minority]}%)')
print(f'Imbalance ratio: {ratio:.1f}:1')
print()

if ratio > 10:
    verdict = 'SEVERE — class_weight="balanced" required, report F1-macro not accuracy'
elif ratio > 3:
    verdict = 'MODERATE — use class_weight="balanced" and stratified CV folds'
else:
    verdict = 'MANAGEABLE — standard training OK, still monitor per-class F1'
print(f'Verdict: {verdict}')

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.barh(RATINGS[::-1], [overall.get(r, 0) for r in RATINGS[::-1]],
        color=[COLORS[r] for r in RATINGS[::-1]], edgecolor='white', height=0.5)
for r in RATINGS:
    cnt = overall.get(r, 0)
    ax.text(cnt + overall.max() * 0.008, RATINGS[::-1].index(r),
            f'{cnt:,}  ({overall_pct[r]}%)', va='center', fontsize=9)
ax.set_xlabel('# hours')
ax.set_title(f'Class imbalance: {ratio:.1f}:1  ({majority} vs {minority})')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_xlim(0, overall.max() * 1.3)
plt.tight_layout()
plt.show()

## 6. Per-break rating distributions

Same swell, same wind — but tide preference differs per break. Does that shift the rating mix?

In [ ]:
per_break = (
    df.groupby(['break', 'rating']).size()
    .unstack(fill_value=0)
    .reindex(columns=RATINGS, fill_value=0)
)
per_break_pct = per_break.div(per_break.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bottom  = np.zeros(len(per_break_pct))
xlabels = [BREAK_LABELS[b] for b in per_break_pct.index]
for r in RATINGS:
    vals = per_break_pct[r].values
    axes[0].bar(xlabels, vals, bottom=bottom, color=COLORS[r],
                label=r, edgecolor='white', linewidth=0.5)
    for i, (v, b) in enumerate(zip(vals, bottom)):
        if v > 4:
            axes[0].text(i, b + v/2, f'{v:.0f}%',
                         ha='center', va='center', fontsize=9,
                         color='white', fontweight='bold')
    bottom += vals
axes[0].set_ylabel('% of hours')
axes[0].set_title('Rating mix per break')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].set_ylim(0, 112)

sns.heatmap(
    per_break.rename(index=BREAK_LABELS),
    annot=True, fmt='d', cmap='YlOrRd',
    linewidths=0.5, ax=axes[1], cbar_kws={'label': '# hours'}
)
axes[1].set_title('Hour counts: break × rating')
axes[1].set_ylabel('')

plt.suptitle('Per-Break Rating Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Per-break % breakdown:')
print(per_break_pct.rename(index=BREAK_LABELS).round(1).to_string())

## 7. Rating by hour of day

SD's sea breeze is predictable: calm and offshore at dawn, onshore by early afternoon. GOOD hours should cluster in the morning window. This validates that `hour_of_day` needs to be a model feature.

In [ ]:
hourly = (
    df.groupby(['pacific_hour', 'rating']).size()
    .unstack(fill_value=0)
    .reindex(columns=RATINGS, fill_value=0)
)
hourly_pct = hourly.div(hourly.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Stacked % bar
bottom = np.zeros(len(hourly_pct))
for r in RATINGS:
    vals = hourly_pct[r].values
    axes[0].bar(hourly_pct.index, vals, bottom=bottom,
                color=COLORS[r], label=r, width=0.85)
    bottom += vals
axes[0].axvspan(4.5,  9.5, alpha=0.08, color='green')
axes[0].axvspan(12.5, 19.5, alpha=0.08, color='red')
axes[0].text(7,  103, 'dawn window', ha='center', color='#27ae60', fontsize=9, fontweight='bold')
axes[0].text(16, 103, 'sea breeze',  ha='center', color='#c0392b', fontsize=9, fontweight='bold')
axes[0].set_ylabel('% of hours')
axes[0].set_title('Rating distribution by hour of day')
axes[0].legend(loc='upper right', fontsize=9)

# Rideable hours count
good_hrs = hourly.get('GOOD', 0) + hourly.get('FAIR', 0)
axes[1].plot(hourly.index, hourly.get('GOOD', pd.Series(0, index=hourly.index)),
             color='#2ecc71', linewidth=2.5, label='GOOD')
axes[1].fill_between(hourly.index,
                     hourly.get('GOOD', pd.Series(0, index=hourly.index)),
                     alpha=0.15, color='#2ecc71')
axes[1].plot(hourly.index, good_hrs, color='#f39c12', linewidth=1.5,
             linestyle='--', label='GOOD + FAIR')
axes[1].axvspan(4.5,  9.5, alpha=0.08, color='green')
axes[1].axvspan(12.5, 19.5, alpha=0.08, color='red')
axes[1].set_ylabel('# hours (180 days × 3 breaks)')
axes[1].set_title('Rideable hours by time of day')
axes[1].legend(fontsize=9)
axes[1].set_xticks(range(24))
axes[1].set_xticklabels([f'{h:02d}h' for h in range(24)], rotation=45, fontsize=8)
axes[1].set_xlabel('Hour of day (Pacific Time, approx UTC−8)')

plt.suptitle("SD diurnal wind cycle — dawn is almost always best", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

peak = int(hourly.get('GOOD', pd.Series(0, index=hourly.index)).idxmax())
print(f'Peak GOOD hour: {peak:02d}h Pacific')

## 8. Pairwise break agreement

How often do two breaks get the same overall rating at the same hour? Low agreement = wind and tide differences matter and break-specific features carry real signal.

In [ ]:
pivot = (
    df.pivot_table(index='hour_utc', columns='break', values='rating', aggfunc='first')
    [BREAKS].dropna()
)

agree_matrix = pd.DataFrame(index=BREAKS, columns=BREAKS, dtype=float)
for a in BREAKS:
    for b in BREAKS:
        agree_matrix.loc[a, b] = (pivot[a] == pivot[b]).mean() * 100

all_agree_pct = pivot.apply(lambda r: r.nunique() == 1, axis=1).mean() * 100
print(f'All 3 breaks same rating:  {all_agree_pct:.1f}%')
print(f'At least one differs:      {100 - all_agree_pct:.1f}%')

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(
    agree_matrix.rename(index=BREAK_LABELS, columns=BREAK_LABELS).astype(float),
    annot=True, fmt='.1f', cmap='Blues', vmin=40, vmax=100,
    linewidths=0.5, ax=ax, cbar_kws={'label': '% hours same rating'}
)
ax.set_title('Pairwise rating agreement between breaks (%)')
plt.tight_layout()
plt.show()

## 9. Key findings

In [ ]:
print('LABEL ANALYSIS — KEY FINDINGS')
print('=' * 50)

print('\n1. OVERALL DISTRIBUTION')
for r in RATINGS:
    print(f'   {r:>5s}: {overall[r]:>7,} hrs  ({overall_pct[r]:.1f}%)')

print(f'\n2. CLASS IMBALANCE: {ratio:.1f}:1')
print(f'   {verdict}')

print('\n3. PER-BREAK')
for brk in BREAKS:
    row   = per_break_pct.loc[brk]
    parts = '  '.join(f'{r}: {row[r]:.0f}%' for r in RATINGS)
    print(f'   {BREAK_LABELS[brk]:20s}  {parts}')

spread        = per_break_pct.max() - per_break_pct.min()
most_variable = spread.idxmax()
print(f'\n   Most variable label: {most_variable} ({spread[most_variable]:.1f}pp spread)')

print(f'\n4. INTER-BREAK AGREEMENT: {all_agree_pct:.1f}% same rating')
print(f'   {100 - all_agree_pct:.1f}% differ → break_id is a useful feature')

print(f'\n5. TIME OF DAY: peak GOOD hour is {peak:02d}h Pacific')
print('   → hour_of_day must be in the feature set')